# Tutorial 1: Data Preprocessing and Tokenization

Welcome to scGPT-mini! This tutorial covers **Phase 1: Foundation & Data Processing**.

## Learning Objectives

By the end of this tutorial, you will understand:
1. How single-cell RNA-seq data is preprocessed
2. How genes are tokenized for transformer models
3. How to create vocabularies and dataloaders
4. How masked language modeling (MLM) works

## Prerequisites

```bash
pip install scanpy torch numpy pandas scikit-learn
```

## Part 1: Loading Single-Cell RNA-seq Data

We'll use the PBMC 3k dataset from 10x Genomics, which contains ~2,700 peripheral blood mononuclear cells.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load PBMC 3k dataset
adata = sc.datasets.pbmc3k()

print(f"Dataset shape: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"Data type: {type(adata.X)}")
print(f"\nFirst few gene names: {adata.var_names[:10].tolist()}")

### What is AnnData?

AnnData is the standard data structure for single-cell analysis:
- `adata.X`: Expression matrix (cells × genes)
- `adata.obs`: Cell metadata (observations)
- `adata.var`: Gene metadata (variables)
- `adata.obsm`: Multi-dimensional cell annotations (e.g., embeddings)

Let's explore the data:

In [ ]:
# Check data sparsity
if hasattr(adata.X, 'toarray'):
    data_array = adata.X.toarray()
else:
    data_array = adata.X

sparsity = (data_array == 0).sum() / data_array.size
print(f"Data sparsity: {sparsity:.2%}")
print(f"Average genes per cell: {(data_array > 0).sum(axis=1).mean():.0f}")
print(f"Average UMI counts per cell: {data_array.sum(axis=1).mean():.0f}")

## Part 2: Preprocessing Pipeline

Single-cell data requires preprocessing before modeling:

1. **Quality Control**: Filter low-quality cells and genes
2. **Normalization**: Account for sequencing depth differences
3. **Log Transformation**: Stabilize variance
4. **Feature Selection**: Select highly variable genes (HVGs)
5. **Binning** (optional): Discretize expression values

Let's do this step by step:

### Step 1: Quality Control

In [ ]:
from scgpt_mini.data import filter_genes, filter_cells

print("Before filtering:")
print(f"  Cells: {adata.n_obs}, Genes: {adata.n_vars}")

# Filter genes expressed in < 3 cells
adata = filter_genes(
    adata,
    min_cells=3,
    min_counts=10,
    inplace=False
)

# Filter cells with < 200 or > 10,000 genes
adata = filter_cells(
    adata,
    min_genes=200,
    max_genes=10000,
    inplace=False
)

print("\nAfter filtering:")
print(f"  Cells: {adata.n_obs}, Genes: {adata.n_vars}")

### Step 2: Normalization

In [ ]:
from scgpt_mini.data import normalize_total

# Normalize to 10,000 counts per cell
adata = normalize_total(adata, target_sum=1e4, inplace=False)

# Check normalization
data_array = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
print(f"Average total counts per cell: {data_array.sum(axis=1).mean():.0f}")
print(f"Std of total counts: {data_array.sum(axis=1).std():.2f}")

### Step 3: Log Transformation

In [ ]:
from scgpt_mini.data import log_transform

# Apply log1p: log(x + 1)
adata = log_transform(adata, inplace=False)

# Visualize distribution before and after
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before log (from raw data)
raw_adata = sc.datasets.pbmc3k()
raw_data = raw_adata.X.toarray() if hasattr(raw_adata.X, 'toarray') else raw_adata.X
axes[0].hist(raw_data.flatten(), bins=50, range=(0, 100), edgecolor='black')
axes[0].set_xlabel('Raw Expression')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Before Log Transformation')

# After log
log_data = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
axes[1].hist(log_data.flatten(), bins=50, edgecolor='black')
axes[1].set_xlabel('Log Expression')
axes[1].set_ylabel('Frequency')
axes[1].set_title('After Log Transformation')

plt.tight_layout()
plt.show()

print("Log transformation stabilizes variance across expression ranges")

### Step 4: Highly Variable Gene (HVG) Selection

In [ ]:
from scgpt_mini.data import select_hvg

# Select top 500 HVGs
n_hvgs = 500
adata = select_hvg(adata, n_top_genes=n_hvgs, inplace=False)

print(f"Selected {adata.n_vars} highly variable genes")
print(f"\nTop 10 HVGs: {adata.var_names[:10].tolist()}")

### All-in-One: Using `preprocess_adata()`

For convenience, scGPT-mini provides a single function:

In [ ]:
from scgpt_mini.data import preprocess_adata

# Reload data
adata = sc.datasets.pbmc3k()

# Preprocess in one go
adata_processed = preprocess_adata(
    adata,
    filter_gene_by_counts=10,
    filter_cell_by_genes=200,
    normalize_total_target=1e4,
    log1p=True,
    subset_hvg=500,
    binning=False,
    inplace=False,
)

print(f"Preprocessed: {adata_processed.n_obs} cells x {adata_processed.n_vars} genes")

## Part 3: Gene Tokenization

Transformers operate on discrete tokens, not continuous values. We need to:
1. Create a **vocabulary** mapping gene names to IDs
2. **Tokenize** each cell's genes
3. Handle **special tokens** (PAD, CLS, EOC)

This is analogous to how BERT tokenizes words in NLP!

### Creating a Gene Vocabulary

In [ ]:
from scgpt_mini.tokenizer import GeneVocab

# Option 1: Create from our genes
gene_list = adata_processed.var_names.tolist()
vocab_custom = GeneVocab(gene_list)

print(f"Custom vocabulary size: {len(vocab_custom)}")
print(f"Special tokens: {vocab_custom.DEFAULT_SPECIAL_TOKENS}")
print(f"\nExample mappings:")
for gene in gene_list[:5]:
    print(f"  {gene} -> {vocab_custom.get_id(gene)}")

# Option 2: Load default vocabulary (recommended)
vocab = GeneVocab.from_json("../scgpt_mini/tokenizer/default_vocab.json")
print(f"\nDefault vocabulary size: {len(vocab)}")

### Understanding Special Tokens

- `<pad>` (ID 0): Padding for variable-length sequences
- `<cls>` (ID 1): Classification token (like BERT's [CLS])
- `<eoc>` (ID 2): End of cell marker

### Tokenizing Cells

In [ ]:
from scgpt_mini.tokenizer import tokenize_batch

# Get data matrix and gene names
data_matrix = adata_processed.X.toarray() if hasattr(adata_processed.X, 'toarray') else adata_processed.X
gene_names = adata_processed.var_names.values

# Tokenize
tokenized_data = tokenize_batch(
    data=data_matrix,
    gene_names=gene_names,
    vocab=vocab,
    append_cls=True,
    include_zero_genes=False,  # Only non-zero genes
    return_pt=True,  # Return PyTorch tensors
)

print(f"Tokenized {len(tokenized_data)} cells")
print(f"\nExample cell:")
genes, values = tokenized_data[0]
print(f"  Gene IDs shape: {genes.shape}")
print(f"  Values shape: {values.shape}")
print(f"  First 5 gene IDs: {genes[:5]}")
print(f"  First 5 values: {values[:5]}")

### Visualizing Tokenization

Let's decode the first cell to understand the tokenization:

In [ ]:
# Decode first cell
genes, values = tokenized_data[0]

print("First 10 tokens:")
print(f"{'Token ID':<10} {'Gene Name':<15} {'Expression':<12}")
print("-" * 40)

for i in range(min(10, len(genes))):
    gene_id = genes[i].item()
    if gene_id == 1:  # CLS token
        gene_name = "<CLS>"
    else:
        gene_name = vocab.get_gene(gene_id)
    expr = values[i].item()
    print(f"{gene_id:<10} {gene_name:<15} {expr:<12.4f}")

## Part 4: Creating DataLoaders

PyTorch DataLoaders batch samples and apply transformations. For scGPT-mini, we need:
1. **Padding**: Make all sequences the same length
2. **Masking**: Create attention masks
3. **MLM masking**: Randomly mask tokens for pretraining

In [ ]:
from scgpt_mini.data import create_dataloader

# Create DataLoader with MLM masking
dataloader = create_dataloader(
    tokenized_data=tokenized_data,
    vocab=vocab,
    batch_size=32,
    max_len=1001,  # Max genes + 1 for CLS
    shuffle=True,
    apply_masking=True,
    mask_ratio=0.15,  # Mask 15% of tokens
)

print(f"Created DataLoader with {len(dataloader)} batches")
print(f"Batch size: 32")
print(f"Total samples: {len(dataloader.dataset)}")

### Understanding a Batch

In [ ]:
# Get one batch
batch = next(iter(dataloader))

genes = batch['genes']
values = batch['values']
attention_mask = batch['attention_mask']
masked_values = batch['masked_values']
mask_positions = batch['mask_positions']
target_values = batch['target_values']

print("Batch contents:")
print(f"  genes: {genes.shape} - Gene IDs")
print(f"  values: {values.shape} - Original expression values")
print(f"  attention_mask: {attention_mask.shape} - Valid positions (1) vs padding (0)")
print(f"  masked_values: {masked_values.shape} - Values with masking applied")
print(f"  mask_positions: {mask_positions.shape} - Which positions are masked")
print(f"  target_values: {target_values.shape} - Ground truth for masked positions")

# Check masking percentage
n_masked = mask_positions.sum().item()
n_valid = attention_mask.sum().item()
mask_pct = n_masked / n_valid
print(f"\nMasking statistics:")
print(f"  Valid tokens: {n_valid}")
print(f"  Masked tokens: {n_masked}")
print(f"  Mask ratio: {mask_pct:.2%}")

## Part 5: Masked Language Modeling (MLM)

MLM is the pretraining objective for scGPT-mini, inspired by BERT:

1. **Randomly mask** 15% of gene expression values
2. **Train model** to predict masked values
3. Model learns **gene relationships** and **biological patterns**

### Visualizing MLM

In [ ]:
# Take first sample from batch
idx = 0
sample_genes = genes[idx]
sample_values = values[idx]
sample_masked = masked_values[idx]
sample_mask = mask_positions[idx]
sample_targets = target_values[idx]

# Find masked positions
masked_indices = sample_mask.nonzero(as_tuple=True)[0][:5]  # First 5 masked

print("Masked Language Modeling Example:")
print(f"\n{'Position':<10} {'Gene':<15} {'Original':<12} {'Masked':<12} {'Target':<12}")
print("-" * 65)

for pos in masked_indices:
    gene_id = sample_genes[pos].item()
    if gene_id > 2:  # Not special token
        gene_name = vocab.get_gene(gene_id)
        original = sample_values[pos].item()
        masked = sample_masked[pos].item()
        target = sample_targets[pos].item()
        print(f"{pos.item():<10} {gene_name:<15} {original:<12.4f} {masked:<12.4f} {target:<12.4f}")

## Summary

In this tutorial, you learned:

✅ **Data Preprocessing**:
- Quality control (filtering)
- Normalization and log transformation
- Highly variable gene selection

✅ **Tokenization**:
- Creating gene vocabularies
- Converting cells to token sequences
- Special tokens (PAD, CLS, EOC)

✅ **DataLoaders**:
- Batching and padding
- Attention masks
- MLM masking for pretraining

✅ **MLM Objective**:
- How models learn from masked genes
- Connection to BERT and NLP

## Next Steps

Continue to **Tutorial 2: Pretraining** to learn how to train the transformer model!

## Exercises

Try these to deepen your understanding:

1. **Change HVG count**: Rerun with 1000 HVGs. How does vocabulary size affect memory?
2. **Different mask ratios**: Try 5%, 30%, 50%. How does this affect learning?
3. **Custom dataset**: Load your own scRNA-seq data and preprocess it
4. **Binning**: Enable binning and visualize binned vs continuous values
5. **Vocabulary analysis**: Which genes appear most frequently in the dataset?